# 02 - Arity Filtration and the Abstract Reduction System

> **Claim-typing legend**
> - **[Pinned]**: reproduced from a named local script, benchmark, or cryptographic receipt (`code/*.json`).
> - **[Proved]**: established by a machine-checked Lean 4 theorem with reported kernel axioms.
> - **[Declared]**: a grammar, carrier alphabet, cost, or objective chosen by the model.
> - **[Open]**: an obligation for which this package supplies no proof or exhaustive decision.

*Redex Matching, Substitution Modulo Group Action, and Finite Termination*

---

## 1. Abstract Reduction Systems (ARS)

In mathematical deduction, rewriting is modeled by an **Abstract Reduction System** $(A, \to)$, where $A$ is the set of symbolic terms and $\to \subseteq A \times A$ is the reduction relation.

When language models perform chain-of-thought (CoT) reasoning, they frequently fall into infinite loops or circular rewrites ($t_1 \to t_2 \to \dots \to t_1$) because token generation lacks cycle detection.

The Symbol Game resolves this via:
1. **Redex-Keyed Identification**: Every reduction step is keyed by an exact redex pattern $(l \to r)$ and substitution $\sigma$.
2. **Action Modulo Symmetric Group $S_n$**: Relational terms are normalized under carrier permutations so isomorphic states share the same canonical key.
3. **Strict Arity Filtration**: Reductions must strictly decrease term weight or contract the carrier defect:
   $$W(t_{k+1}) < W(t_k)$$


In [ ]:
def reduce_once(term: str):
    rules = (("f(g(x))", "f(x)"), ("f(x)", "x"))
    for left, right in rules:
        if term == left:
            return right
    return None


start = "f(g(x))"
middle = reduce_once(start)
end = reduce_once(middle)
assert middle == "f(x)"
assert end == "x"
assert reduce_once(end) is None
print(start, "-->", middle, "-->", end)

## Anchors

- Former notebook: `02_arity_filtration_and_ars_rewriting`
- The code checks one finite example. It is not a general proof.

## Exercise

Change one declared input in the code cell. The check should fail for the wrong object and pass for the finite example written here.


---
## Integrated Laboratory: Genesis.02: Response Evaluation, Structural Quotients & Confluence
`AuthorityEffect: None.`

Consolidated laboratory covering Genesis Stages 06 through 12.
`[Declared]` Objective: Evaluate candidate responses deterministically, quotient isomorphic representations, and verify Church-Rosser confluence.


In [ ]:
# [Pinned] Environment Preflight
import hashlib
import sys
from typing import Dict, List, Set, Tuple

assert sys.version_info >= (3, 10), "Python 3.10+ required"
print(
    f"[Preflight OK] Running under Python {sys.version.split()[0]} with exact rational capability."
)

## Part 1: Minimal Candidate Response Evaluation
`[Declared]` Evaluate candidate responses deterministically with typed error codes.


In [ ]:
# --- PREDICTION GATE 1 ---
# What return status occurs when a well-formed candidate is evaluated?
# Options: 'response_deterministic', 'ERR_MALFORMED', 'unhandled_exception'
prediction = "response_deterministic"


def evaluate_candidate(payload: str) -> str:
    if not payload:
        return "ERR_EMPTY_PAYLOAD"
    if not (payload.startswith("<") and payload.endswith(">")):
        return "ERR_MALFORMED_BOUNDARY"
    return "response_deterministic"


status = evaluate_candidate("<atom_test>")
assert status == prediction

# [Tested] Adversarial Falsifier: reject malformed input without crash
assert evaluate_candidate("bad_string") == "ERR_MALFORMED_BOUNDARY"
assert evaluate_candidate("") == "ERR_EMPTY_PAYLOAD"
print("[Candidate Response OK] Verified deterministic response evaluation.")

## Part 2: Structural Isomorphism Quotienting
`[Declared]` Map isomorphic terms to a unique canonical normal form.


In [ ]:
# --- PREDICTION GATE 2 ---
# Do two multisets with identical elements in different order produce identical canonical forms?
# Options: 'quotient_canonical', 'order_dependent', 'hash_collision'
prediction = "quotient_canonical"


def canonicalize_multiset(elements: List[str]) -> Tuple[str, str]:
    # Sort elements deterministically
    sorted_elems = sorted(elements)
    canonical_repr = "<" + ",".join(sorted_elems) + ">"
    h = hashlib.sha256(canonical_repr.encode("utf-8")).hexdigest()
    return ("quotient_canonical", h)


status1, h1 = canonicalize_multiset(["b", "a", "c"])
status2, h2 = canonicalize_multiset(["c", "a", "b"])
assert status1 == prediction
assert h1 == h2, f"Hashes differ: {h1} vs {h2}"

# [Tested] Adversarial Falsifier: distinct multisets produce distinct hashes
_, h3 = canonicalize_multiset(["a", "b", "d"])
assert h1 != h3
print(f"[Quotient OK] Verified canonical normal form hash: {h1[:16]}...")

## Part 3: Trace Algebra Confluence (Newman Diamond)
`[Declared]` Verify local confluence on term rewriting branches.


In [ ]:
# --- PREDICTION GATE 3 ---
# When two rewrite branches diverge from a common ancestor, do they reconverge at a common normal form?
# Options: 'church_rosser_confluent', 'divergent', 'looping'
prediction = "church_rosser_confluent"

# Term rewrite system:
# S -> A (branch 1)
# S -> B (branch 2)
# A -> C, B -> C (reconvergence)
rewrites = {"S": ["A", "B"], "A": ["C"], "B": ["C"], "C": []}


def verify_local_confluence(r_map: Dict[str, List[str]], root: str) -> str:
    branches = r_map.get(root, [])
    if len(branches) < 2:
        return "trivially_confluent"

    # Find normal forms from each branch
    def find_nf(node: str) -> Set[str]:
        succ = r_map.get(node, [])
        if not succ:
            return {node}
        res = set()
        for s in succ:
            res.update(find_nf(s))
        return res

    nf_b1 = find_nf(branches[0])
    nf_b2 = find_nf(branches[1])
    if nf_b1 == nf_b2 and len(nf_b1) == 1:
        return "church_rosser_confluent"
    return "divergent"


conf_status = verify_local_confluence(rewrites, "S")
assert conf_status == prediction

# [Tested] Adversarial Falsifier: non-confluent system is detected
bad_rewrites = {"S": ["A", "B"], "A": ["C1"], "B": ["C2"], "C1": [], "C2": []}
assert verify_local_confluence(bad_rewrites, "S") == "divergent"
print("[Confluence OK] Verified Church-Rosser diamond property.")